In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Database + Schema Set-Up
session.sql("USE DATABASE AMAZON_DATASALES").collect()
session.sql("USE SCHEMA PUBLIC").collect()

print("Connected. Current context set.")

In [ ]:
import pandas as pd

df = pd.read_csv("amazon_sales_dataset.csv")
df.head()

In [ ]:
sp_df = session.create_dataframe(df)

sp_df.write.mode("overwrite").save_as_table("AMAZON_SALES_RAW")

print("Uploaded to AMAZON_SALES_RAW")

In [ ]:
%%sql -r dataframe_1
USE DATABASE AMAZON_DATASALES;
USE SCHEMA PUBLIC;

-- 1) CLEAN: standardize names + enforce DATE type
CREATE OR REPLACE TABLE AMAZON_SALES_CLEAN AS
SELECT
  "order_id"::NUMBER(38,0)                     AS order_id,
  TO_DATE("order_date")                        AS order_date,
  "product_id"::NUMBER(38,0)                   AS product_id,
  "product_category"::STRING                   AS product_category,
  "price"::FLOAT                               AS price,
  "discount_percent"::FLOAT                    AS discount_percent,
  "quantity_sold"::NUMBER(38,0)                AS quantity_sold,
  "customer_region"::STRING                    AS customer_region,
  "payment_method"::STRING                     AS payment_method,
  "rating"::FLOAT                              AS rating,
  "review_count"::NUMBER(38,0)                 AS review_count,
  "discounted_price"::FLOAT                    AS discounted_price,
  "total_revenue"::FLOAT                       AS total_revenue
FROM AMAZON_SALES_RAW
WHERE TO_DATE("order_date") IS NOT NULL
  AND "total_revenue" IS NOT NULL;

-- 2) DAILY TARGET SERIES
CREATE OR REPLACE TABLE TS_DAILY AS
SELECT
  order_date AS ds,
  SUM(total_revenue) AS y_revenue,
  SUM(quantity_sold) AS y_units,
  COUNT(*) AS y_orders
FROM AMAZON_SALES_CLEAN
GROUP BY order_date
ORDER BY order_date;

-- 3) DAILY EXOGENOUS FEATURES (SARIMAX predictors)
CREATE OR REPLACE TABLE TS_DAILY_EXOG AS
SELECT
  order_date AS ds,
  AVG(discount_percent) AS avg_discount_pct,
  AVG(discounted_price) AS avg_discounted_price,

  AVG(IFF(product_category='Electronics', 1, 0)) AS share_electronics,
  AVG(IFF(product_category='Fashion', 1, 0)) AS share_fashion,
  AVG(IFF(product_category='Home & Kitchen', 1, 0)) AS share_home_kitchen,

  AVG(IFF(customer_region='North America', 1, 0)) AS share_na,
  AVG(IFF(customer_region='Europe', 1, 0)) AS share_europe,

  DAYOFWEEKISO(order_date) AS dow_iso,
  MONTH(order_date) AS month_num,
  IFF(DAYOFWEEKISO(order_date) IN (6,7), 1, 0) AS is_weekend
FROM AMAZON_SALES_CLEAN
GROUP BY order_date
ORDER BY order_date;

-- 4) MODEL-READY VIEW
CREATE OR REPLACE VIEW V_MODEL_DAILY_REVENUE AS
SELECT
  t.ds,
  t.y_revenue AS y,
  t.y_orders,
  t.y_units,
  x.avg_discount_pct,
  x.avg_discounted_price,
  x.share_electronics,
  x.share_fashion,
  x.share_home_kitchen,
  x.share_na,
  x.share_europe,
  x.dow_iso,
  x.month_num,
  x.is_weekend
FROM TS_DAILY t
LEFT JOIN TS_DAILY_EXOG x
  ON t.ds = x.ds
ORDER BY t.ds;



In [ ]:
%%sql -r dataframe_4
-- 5) VALIDATION CHECKS
SELECT COUNT(*) AS clean_rows FROM AMAZON_SALES_CLEAN;

SELECT COUNT(*) AS n_days, MIN(ds) AS min_day, MAX(ds) AS max_day
FROM TS_DAILY;

SELECT * FROM V_MODEL_DAILY_REVENUE ORDER BY ds LIMIT 5;

In [ ]:
# Connect to Snowflake session
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

# Pull daily revenue table
df = session.table("TS_DAILY").to_pandas()

df = df.sort_values("DS")
df = df.set_index("DS")

df.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
plt.plot(df.index, df["Y_REVENUE"])
plt.title("Daily Total Revenue")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.show()


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df["Y_REVENUE"], model="additive", period=7)

decomposition.plot()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

warnings.filterwarnings("ignore")

adf_stat, adf_p, *_ = adfuller(df["Y_REVENUE"])
kpss_stat, kpss_p, *_ = kpss(df["Y_REVENUE"], regression="c")

print(f"ADF:  stat={adf_stat:.4f}, p={adf_p:.4f}")
print(f"KPSS: stat={kpss_stat:.4f}, p={kpss_p:.4f}")

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(12,4))
plot_acf(df["Y_REVENUE"], lags=40)
plt.show()

plt.figure(figsize=(12,4))
plot_pacf(df["Y_REVENUE"], lags=40)
plt.show()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

df = df.asfreq("D")
ets_model = ExponentialSmoothing(
    df["Y_REVENUE"],
    trend="add",
    seasonal="add",
    seasonal_periods=7
).fit()

print(ets_model.summary())

df["ETS_Fitted"] = ets_model.fittedvalues

plt.figure(figsize=(14,6))
plt.plot(df["Y_REVENUE"], label="Actual")
plt.plot(df["ETS_Fitted"], label="ETS Fitted")
plt.legend()
plt.show()

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

df = session.table("V_MODEL_DAILY_REVENUE").to_pandas()
df = df.sort_values("DS")
df = df.set_index("DS")
df = df.asfreq("D")

df.head()

In [ ]:
train = df.iloc[:-30]
test = df.iloc[-30:]

y_train = train["Y"]
y_test = test["Y"]

X_train = train.drop(columns=["Y"])
X_test = test.drop(columns=["Y"])

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(y_train, order=(1,0,1)).fit()
arima_forecast = arima_model.forecast(steps=30)

In [ ]:
sarima_model = ARIMA(
    y_train,
    order=(1,0,1),
    seasonal_order=(1,0,1,7)
).fit()

sarima_forecast = sarima_model.forecast(steps=30)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1,0,1),
    seasonal_order=(1,0,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit()

sarimax_forecast = sarimax_model.forecast(steps=30, exog=X_test)

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return round(mae,2), round(rmse,2), round(mape,2)

print("ARIMA  :", evaluate(y_test, arima_forecast))
print("SARIMA :", evaluate(y_test, sarima_forecast))
print("SARIMAX:", evaluate(y_test, sarimax_forecast))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))
plt.plot(y_test, label="Actual")
plt.plot(arima_forecast, label="ARIMA")
plt.plot(sarima_forecast, label="SARIMA")
plt.plot(sarimax_forecast, label="SARIMAX")
plt.legend()
plt.title("30-Day Forecast Comparison")
plt.show()

In [ ]:
def walk_forward(model_type):
    errors = []

    for i in range(30):
        train_split = df.iloc[:-(30-i)]
        test_split = df.iloc[-(30-i)]

        y_train = train_split["Y"]

        if model_type == "arima":
            model = ARIMA(y_train, order=(1,0,1)).fit()
            pred = model.forecast()[0]

        elif model_type == "sarima":
            model = ARIMA(
                y_train,
                order=(1,0,1),
                seasonal_order=(1,0,1,7)
            ).fit()
            pred = model.forecast()[0]

        elif model_type == "sarimax":
            X_train = train_split.drop(columns=["Y"])
            X_test = df.iloc[[-(30-i)]].drop(columns=["Y"])

            model = SARIMAX(
                y_train,
                exog=X_train,
                order=(1,0,1),
                seasonal_order=(1,0,1,7)
            ).fit(disp=False)

            pred = model.forecast(exog=X_test)[0]

        errors.append(abs(test_split["Y"] - pred))

    return round(np.mean(errors),2)

print("Walk-Forward MAE")
print("ARIMA :", walk_forward("arima"))
print("SARIMA:", walk_forward("sarima"))
print("SARIMAX:", walk_forward("sarimax"))

In [ ]:
import pandas as pd

holdout = {
    "ARIMA": evaluate(y_test, arima_forecast),
    "SARIMA": evaluate(y_test, sarima_forecast),
    "SARIMAX": evaluate(y_test, sarimax_forecast),
}

wf_mae = {
    "ARIMA": 5927.34,
    "SARIMA": 5957.80,
    "SARIMAX": 2059.29
}

results = pd.DataFrame([
    {"Model": m, "MAE": holdout[m][0], "RMSE": holdout[m][1], "MAPE_%": holdout[m][2], "WF_MAE": wf_mae[m]}
    for m in ["ARIMA","SARIMA","SARIMAX"]
])

results

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

def save_forecast_to_snowflake(model_name, y_true, y_pred):
    out = pd.DataFrame({
        "DS": y_true.index,
        "Y_ACTUAL": y_true.values,
        "Y_PRED": pd.Series(y_pred, index=y_true.index).values,
        "MODEL": model_name
    })
    session.write_pandas(
        out,
        table_name=f"FORECAST_HOLDOUT_{model_name}",
        auto_create_table=True,
        overwrite=True
    )
    return out

save_forecast_to_snowflake("ARIMA", y_test, arima_forecast)
save_forecast_to_snowflake("SARIMA", y_test, sarima_forecast)
save_forecast_to_snowflake("SARIMAX", y_test, sarimax_forecast)

In [ ]:
session.write_pandas(
    results,
    table_name="PHASE4_MODEL_METRICS",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
# Explainability (feature impact)
coef = sarimax_model.params

coef_df = coef.reset_index()
coef_df.columns = ["Variable", "Coefficient"]

# define exogenous variable names
exog_columns = X_train.columns.tolist()

exog_coefs = coef_df[coef_df["Variable"].isin(exog_columns)]

# Absolute Importance Ranking
exog_coefs["Abs_Coefficient"] = exog_coefs["Coefficient"].abs()
exog_coefs = exog_coefs.sort_values("Abs_Coefficient", ascending=False)

exog_coefs

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.barh(exog_coefs["Variable"], exog_coefs["Coefficient"])
plt.title("SARIMAX Exogenous Feature Impact on Revenue")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
session.write_pandas(
    exog_coefs,
    table_name="PHASE5_FEATURE_IMPORTANCE",
    auto_create_table=True,
    overwrite=True
)

In [ ]:
# Create 30 future dates
future_dates = pd.date_range(
    start=df.index[-1] + pd.Timedelta(days=1),
    periods=30,
    freq="D"
)

# Create future exogenous dataframe
future_exog = pd.DataFrame(index=future_dates)

# Fill with recent averages (reasonable assumption)
for col in X_train.columns:
    future_exog[col] = df[col].tail(30).mean()

future_exog.head()

future_forecast = sarimax_model.forecast(
    steps=30,
    exog=future_exog
)


In [ ]:
scenario_exog = future_exog.copy()
scenario_exog["AVG_DISCOUNT_PCT"] *= 1.05

scenario_forecast = sarimax_model.forecast(
    steps=30,
    exog=scenario_exog
)

impact = scenario_forecast.sum() - future_forecast.sum()

print("Estimated 30-day revenue impact:", round(impact,2))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))
plt.plot(future_forecast.index, future_forecast, label="Baseline")
plt.plot(scenario_forecast.index, scenario_forecast, label="Scenario: +5% discount", linestyle="--")
plt.title("30-Day Revenue Forecast: Baseline vs Scenario")
plt.xlabel("Date")
plt.ylabel("Forecast Revenue")
plt.legend()
plt.show()